<a href="https://colab.research.google.com/github/dkngit55-glitch/DKNGIT5/blob/main/LHJ_TEST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install crewai langchain-google-genai pandas openpyxl Pillow easyocr

In [2]:
import imaplib, smtplib, email, io, base64, re
import pandas as pd
from email.header import decode_header
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import easyocr

# CrewAI 필수 라이브러리
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool
from google.colab import userdata

# --- [STEP 1] 설정 및 초기화 ---
NAVER_ID = "dkngit5"
NAVER_PW = "HCSHSG79ZBY8"
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# OCR 및 LLM 설정
reader = easyocr.Reader(['ko', 'en'])
my_llm = LLM(model="gemini/gemini-2.5-flash", api_key=GEMINI_API_KEY)

# --- [STEP 2] Helper Functions ---
def safe_decode(header_value):
    if not header_value: return ""
    decoded_list = decode_header(header_value)
    result = ""
    for content, charset in decoded_list:
        if isinstance(content, bytes):
            try: result += content.decode(charset if charset else "utf-8", errors="ignore")
            except: result += content.decode("cp949", errors="ignore")
        else: result += content
    return result

def extract_clean_email(sender_str):
    """'이름 <addr@mail.com>' 형태에서 이메일 주소만 추출"""
    match = re.search(r'[\w\.-]+@[\w\.-]+', sender_str)
    return match.group(0) if match else sender_str

# --- [STEP 3] CrewAI 전용 도구 정의 ---

class FetchEmailsWithOCR(BaseTool):
    name: str = "fetch_unread_emails"
    description: str = "읽지 않은 메일을 가져오고, 이미지/엑셀 내용을 텍스트로 변환 후 읽음 처리합니다."

    def _run(self, query: str = "UNSEEN") -> str:
        mail_data_list = []
        try:
            mail = imaplib.IMAP4_SSL("imap.naver.com")
            mail.login(f"{NAVER_ID}@naver.com", NAVER_PW)
            mail.select("inbox")
            status, messages = mail.search(None, query)
            unread_ids = messages[0].split()

            if not unread_ids:
                mail.logout()
                return "새로운 메일이 없습니다."

            for email_id in unread_ids:
                status, msg_data = mail.fetch(email_id, "(RFC822)")
                for response_part in msg_data:
                    if isinstance(response_part, tuple):
                        msg = email.message_from_bytes(response_part[1])
                        subject = safe_decode(msg.get("Subject"))
                        # 발신자 주소 정제 (중요!)
                        sender_raw = safe_decode(msg.get("From"))
                        sender = extract_clean_email(sender_raw)

                        mail.store(email_id, '+FLAGS', '\\Seen') # 읽음 표시

                        body, attachments_data = "", ""
                        if msg.is_multipart():
                            for part in msg.walk():
                                content_type = part.get_content_type()
                                filename = safe_decode(part.get_filename())
                                if content_type == "text/plain" and not filename:
                                    body = part.get_payload(decode=True).decode('utf-8', errors='ignore')
                                elif filename and filename.lower().endswith(('.xlsx', '.xls')):
                                    df = pd.read_excel(io.BytesIO(part.get_payload(decode=True)))
                                    attachments_data += f"\n[엑셀 데이터 ({filename})]:\n{df.to_string()}\n"
                                elif filename and filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                                    img_bytes = part.get_payload(decode=True)
                                    ocr_result = reader.readtext(img_bytes, detail=0)
                                    attachments_data += f"\n[이미지 OCR 텍스트 ({filename})]: {' '.join(ocr_result)}\n"
                        else:
                            body = msg.get_payload(decode=True).decode('utf-8', errors='ignore')

                        mail_data_list.append({
                            "sender": sender,
                            "subject": subject,
                            "full_content": f"본문: {body}\n첨부데이터분석: {attachments_data}"
                        })

            mail.expunge() # 변경 사항 서버 확정
            mail.logout()
            return str(mail_data_list)
        except Exception as e:
            return f"오류 발생: {e}"

class SendReplyTool(BaseTool):
    name: str = "send_reply_email"
    description: str = "recipient(수신자), subject(제목), body(본문)를 사용하여 답장을 보냅니다."

    def _run(self, recipient: str, subject: str, body: str) -> str:
        try:
            # 주소에 꺾쇠괄호나 이름이 포함되어 있으면 제거
            recipient = extract_clean_email(recipient)
            smtp = smtplib.SMTP_SSL("smtp.naver.com", 465)
            smtp.login(f"{NAVER_ID}@naver.com", NAVER_PW)
            msg = MIMEMultipart()
            msg["From"] = f"{NAVER_ID}@naver.com"
            msg["To"] = recipient
            msg["Subject"] = "Re: " + subject
            msg.attach(MIMEText(body, "plain", "utf-8"))
            smtp.sendmail(f"{NAVER_ID}@naver.com", recipient, msg.as_string())
            smtp.quit()
            return f"✅ {recipient}에게 성공적으로 답장을 보냈습니다."
        except Exception as e:
            return f"❌ {recipient}에게 발송 실패: {e}"

# --- [STEP 4] Agents 정의 (주소 준수 강조) ---

classifier = Agent(
    role='이메일 데이터 선별가',
    goal='광고를 제외하고 실제 사람이 보낸 중요 업무 메일만 필터링한다.',
    backstory='당신은 메일 본문과 이미지 OCR 내용을 보고 실제 회신이 필요한 건인지 판단합니다.',
    llm=my_llm, verbose=True, allow_delegation=False
)

analyst = Agent(
    role='멀티모달 데이터 분석가',
    goal='전달받은 텍스트, 엑셀, 이미지 OCR 데이터를 요약하고 발신자 정보를 유지한다.',
    backstory='당신은 발신자(sender) 주소를 절대로 유실하지 않고 요약본과 함께 전달하는 전문가입니다.',
    llm=my_llm, verbose=True, allow_delegation=False
)

writer = Agent(
    role='비즈니스 소통 매니저',
    goal='분석된 정보를 바탕으로 실제 발신자에게 답장을 작성하고 발송한다.',
    backstory="""당신은 'sender' 필드에 있는 실제 이메일 주소로만 답장을 보냅니다.
    user@example.com 같은 가짜 주소는 절대 사용하지 마세요.
    분석된 이미지/엑셀 내용을 언급하며 친절한 '해요체'로 소통하세요.""",
    tools=[SendReplyTool()], llm=my_llm, verbose=True, allow_delegation=False
)

# --- [STEP 5] Tasks 정의 ---

t1 = Task(
    description="fetch_unread_emails 도구를 호출하여 메일을 가져오세요. 이미지 OCR 내용이 포함됩니다.",
    expected_output="sender, subject, full_content가 포함된 메일 데이터 리스트",
    agent=classifier, tools=[FetchEmailsWithOCR()]
)

t2 = Task(
    description="가져온 데이터 중 답장이 필요한 중요 업무 메일만 골라내세요.",
    expected_output="선별된 메일 정보 (sender 주소 유지 필드 필수)",
    agent=classifier, context=[t1]
)

t3 = Task(
    description="메일 내용을 상세히 분석하세요. 엑셀 수치나 이미지 텍스트 내용을 요약에 반드시 포함하세요.",
    expected_output="각 메일별 분석 요약 및 발신자(sender) 주소 리스트",
    agent=analyst, context=[t2]
)

t4 = Task(
    description="""
    분석된 각 메일의 'sender' 주소로 답장을 보내세요.
    반드시 send_reply_email 도구를 사용하세요.
    - recipient: 분석 데이터의 'sender' 필드값
    - subject: 분석 데이터의 'subject' 필드값
    - body: 정중한 '해요체' 답장
    """,
    expected_output="실제 발신자에게 발송된 최종 결과 보고서",
    agent=writer, context=[t3]
)

# --- [STEP 6] Crew 실행 ---
mail_crew = Crew(
    agents=[classifier, analyst, writer],
    tasks=[t1, t2, t3, t4],
    process=Process.sequential,
    verbose=True
)

print("🚀 OCR 및 주소 인식 강화 시스템 가동...")
print(mail_crew.kickoff())

🚀 OCR 및 주소 인식 강화 시스템 가동...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c9a95336-1ddb-47db-a5dc-bff08f3ac2fc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: fetch_unread_emails 도구를 호출하여 메일을 가져오세요. 이미지 OCR 내용이 포함됩니다.                     │
│  ID: a18a6960-2236-4873-9a95-a4e493ccd017                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 이메일 데이터 선별가                                                                                    │
│                                                                                                                 │
│  Task: fetch_unread_emails 도구를 호출하여 메일을 가져오세요. 이미지 OCR 내용이 포함됩니다.                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: fetch_unread_emails                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 비즈니스 소통 매니저                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ✅ lhj@dong-il.com에게 성공적으로 답장을 보냈습니다.                                                           │
│  ✅ lhj@dong-il.com에게 성공적으로 답장을 보냈습니다.                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      분석된 각 메일의 'sender' 주소로 답장을 보내세요.                                                          │
│      반드시 send_reply_email 도구를 사용하세요.                                                                 │
│      - recipient: 분석 데이터의 'sender' 필드값                                                                 │
│      - subject: 분석 데이터의 'subject' 필드값                                                                  │
│      - body: 정중한 '해요체' 답장                                                                               │
│                                                                                                                 │
│  Agent: 비즈니스 소통 매니저                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c9a95336-1ddb-47db-a5dc-bff08f3ac2fc                                                                       │
│  Final Output: ✅ lhj@dong-il.com에게 성공적으로 답장을 보냈습니다.                                             │
│  ✅ lhj@dong-il.com에게 성공적으로 답장을 보냈습니다.                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ lhj@dong-il.com에게 성공적으로 답장을 보냈습니다.
✅ lhj@dong-il.com에게 성공적으로 답장을 보냈습니다.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯